## pkpy Intro
A tour of the pkpy poker analysis library — Python bindings for the pkcore Rust engine.

In [2]:
import pkpy

### Cards
Parse individual cards and inspect their rank and suit.

In [3]:
c = pkpy.Card.parse("As")
print(f"Card   : {c}")
print(f"Rank   : {c.rank()} (value={c.rank().value()})")
print(f"Suit   : {c.suit()} (symbol={c.suit().symbol()})")
print(f"Cactus Kev u32: {c.as_u32():#010x}")

Card   : A♠
Rank   : A (value=14)
Suit   : ♠ (symbol=♠)
Cactus Kev u32: 0x10008c29


### Deck
Build a full 52-card deck and inspect it.

In [4]:
deck = pkpy.Cards.deck()
print(f"Deck size: {len(deck)}")
print(f"Cards    : {deck}")

Deck size: 52
Cards    : A♠ K♠ Q♠ J♠ T♠ 9♠ 8♠ 7♠ 6♠ 5♠ 4♠ 3♠ 2♠ A♥ K♥ Q♥ J♥ T♥ 9♥ 8♥ 7♥ 6♥ 5♥ 4♥ 3♥ 2♥ A♦ K♦ Q♦ J♦ T♦ 9♦ 8♦ 7♦ 6♦ 5♦ 4♦ 3♦ 2♦ A♣ K♣ Q♣ J♣ T♣ 9♣ 8♣ 7♣ 6♣ 5♣ 4♣ 3♣ 2♣


### Board
Parse community cards and see what's visible at each street.

In [5]:
board = pkpy.Board.parse("As Ks Qh Jd Tc")
print(f"Full board  : {board}")
print(f"Through turn: {board.turn_cards()}")

Full board  : FLOP: A♠ K♠ Q♥, TURN: J♦, RIVER: T♣
Through turn: A♠ K♠ Q♥ J♦


### Combo ranges
A `Combo` is an abstract hand category like `AKs` or `QQ+`.
`Combos` is a range; `.explode()` expands it into every concrete two-card hand.

In [6]:
# Top ~2.5% of starting hands: QQ+, AK
range_str = pkpy.Combos.PERCENT_2_5
combos = pkpy.Combos.parse(range_str)
twos = combos.explode()

print(f"Range         : {range_str}")
print(f"Abstract combos: {len(combos)}")
print(f"Concrete hands : {len(twos)}")
print()

suited   = twos.filter_is_suited()
offsuit  = twos.filter_is_not_suited().filter_is_not_paired()
pairs    = twos.filter_is_paired()

print(f"  Suited  : {len(suited)}")
print(f"  Offsuit : {len(offsuit)}")
print(f"  Pairs   : {len(pairs)}")

Range         : QQ+, AK
Abstract combos: 2
Concrete hands : 34

  Suited  : 4
  Offsuit : 12
  Pairs   : 18


### Outs calculation
Given hole cards and a turn board, find which river cards give each player the win.

In [7]:
# Player 1: A♠ K♥  |  Player 2: 8♦ K♣  (top pair vs. two pair draw)
# Board through the turn: A♣ 8♥ 7♥ 9♠
hc    = pkpy.HoleCards.parse("As Kh 8d Kc")
board = pkpy.Board.parse("Ac 8h 7h 9s")
game  = pkpy.Game(hc, board)

case_evals = game.turn_case_evals()
outs = pkpy.Outs.from_case_evals(case_evals)

print(f"Possible river cards : {len(case_evals)}")
print()
for player in (1, 2):
    n = outs.len_for_player(player)
    cards = outs.get(player)
    winner = " <-- most outs" if outs.is_longest(player) else ""
    print(f"Player {player}: {n} outs — {cards}{winner}")

Possible river cards : 44

Player 1: 42 outs — K♠ Q♠ J♠ T♠ 7♠ 6♠ 5♠ 4♠ 3♠ 2♠ A♥ Q♥ J♥ T♥ 9♥ 6♥ 5♥ 4♥ 3♥ 2♥ A♦ K♦ Q♦ J♦ T♦ 9♦ 7♦ 6♦ 5♦ 4♦ 3♦ 2♦ Q♣ J♣ T♣ 9♣ 7♣ 6♣ 5♣ 4♣ 3♣ 2♣ <-- most outs
Player 2: 2 outs — 8♠ 8♣


### Hand space stats
Quick look at the numbers behind Texas Hold'em.

In [8]:
print(f"Unique 2-card starting hands : {pkpy.unique_2_card_hands():,}")
print(f"Distinct 2-card hand types   : {pkpy.distinct_2_card_hands():,}")
print(f"Unique 5-card hands (52C5)   : {pkpy.unique_5_card_hands():,}")
print(f"Distinct 5-card hand rankings: {pkpy.distinct_5_card_hands():,}")

Unique 2-card starting hands : 1,326
Distinct 2-card hand types   : 169
Unique 5-card hands (52C5)   : 2,598,960
Distinct 5-card hand rankings: 7,462
